### **Silver to Gold: Buliding BI ready tables**

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType,FloatType
import pyspark.sql.functions as F
from pyspark.sql import Row

In [0]:
catalog_name="ecommerce"

### **Products**

In [0]:
#reading products,brands and category tables

df_products = spark.table(f"{catalog_name}.silver.slv_products")
df_brands = spark.table(f"{catalog_name}.silver.slv_brands")
df_category = spark.table(f"{catalog_name}.silver.slv_category")

In [0]:
#create temporary views for the 3 tables
df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands")
df_category.createOrReplaceTempView("v_category")

In [0]:
display(spark.sql ("select * from v_products limit 5"))


In [0]:
display(spark.sql ("select * from v_brands limit 20"))

In [0]:
display(spark.sql ("select * from v_category limit 15"))

In [0]:
#make sure we are on the right catalog

spark.sql(f"USE CATALOG {catalog_name}")

In [0]:
%sql
-- build brandxcategory mapping and write gold table
CREATE OR REPLACE TABLE gold.gld_dim_products AS 

WITH brands_category AS (
  SELECT 
  b.brand_name,
  b.brand_code,
  c.category_name,
  c.category_code
  FROM 
  v_brands b 
  INNER JOIN v_category c 
  ON b.brand_category = c.category_code
)

SELECT 
  p.product_id,
  p.sku,
  p.category_code,
  COALESCE(bc.category_name, 'Not Available') AS category_name,
  p.brand_code,
  COALESCE(bc.brand_name, 'Not Available') AS brand_name,
  p.color,
  p.size,
  p.material,
  p.weight_grams,
  p.length_cm,
  p.width_cm,
  p.height_cm,
  p.rating_count,
  p.ingested_at,
  p._source_file
FROM 
  v_products p
INNER JOIN brands_category bc
ON p.brand_code = bc.brand_code

### **CUSTOMERS**

In [0]:
#map states to regions

#Indian states

indian_regions={
"MH":"West","GJ":"West","RJ":"West",
"KS":"South","TN":"South","TS":"South","AP":"South","KL":"South",
"UP":"West","WB":"West","DL":"West"
}

#Australian States

australia_region ={
    "VIC":"Southeast","WA":"West","NSW":"East", "QLD":"Northeast"
}

#United Kingdom States

uk_region={
    "ENG":"England", "WLS":"Wales", "NIR":"Northern Ireland", "SCT":"Scotland"
}

#US states

US_regions = {
    "MA":"Northeast", "FL":"South", "NJ":"Northeast","CA":"West","NY":"Northeast","TX":"South"
}

#UAE states

UAE_regions={
    "AUH":"Abu Dhabi", "DU":"Dubai", "SHJ":"Sharjah"
}

#canadian states

canada_regions={
    "BC":"West", "AB":"West", "ON":"East", "QC":"East", "NS":"East", "IL":"Other"
}

#singapore states

singapore_regions={
    "SG":"Singapore"
}

# combine into master dictionary

country_state_map={
    "India":indian_regions,
    "Australia":australia_region,
    "United Kingdom":uk_region,
    "United States":US_regions,
    "United Arab Emirates":UAE_regions,
    "Canada":canada_regions,
    "Singapore":singapore_regions
}

country_state_map

In [0]:
#Flatten country_state_map in to list of rows

states_regions = []

for country,states in country_state_map.items():
    for state,region in states.items():
        states_regions.append(Row(country=country,state=state,region=region))
        
states_regions

In [0]:
#Create mapping dataframe
states_regions_df = spark.createDataFrame(states_regions)

states_regions_df.show(truncate=False)


In [0]:
df_silver=spark.table(f'{catalog_name}.silver.slv_customers')
display(df_silver.limit(5))

In [0]:
df_gold=df_silver.join(states_regions_df, on=['country', 'state'], how="left")
df_gold=df_gold.fillna({'region':'Other'})
display(df_gold.limit(5))

In [0]:
df_gold=df_gold.select("customer_id", "phone", "country_code", "country", "state", "region", "_source_file", "ingested_at")
display(df_gold.limit(10))


In [0]:
#write to gold table
df_gold.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog_name}.gold.gld_dim_customers")

### DATE/**CALENDER**

In [0]:
df_silver=spark.table(f'{catalog_name}.silver.slv_date')
display(df_silver.limit(5))

In [0]:
df_gold=df_silver.withColumn("date_id", F.date_format("date", "yyyyMMdd").cast("int"))

#add month name(eg 'January', February etc)
df_gold=df_gold.withColumn("month_name", F.date_format(F.col("date"), "MMMM"))

#add is_weekend column
df_gold=df_gold.withColumn("is_weekend", F.when(F.col("day_name").isin("Saturday", "Sunday"), 1).otherwise(0))

display(df_gold.limit(5))


In [0]:
desired_column_order=["date_id","date","year","month_name","day_name", "is_weekend", "quarter","ingested_at", "_source_file"]

df_gold=df_gold.select(desired_column_order)
display(df_gold.limit(5))


In [0]:
#write to gold layer
df_gold.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog_name}.gold.gld_dim_date")
